# Validation Step 1 — OpenBTAI Preprocessing
## Apply FROZEN Phase 1 pipeline to unseen data

### Key design clarifications

**Role of masks in validation:**
The expert OpenBTAI masks are used **directly** as BSF input.
This isolates: *Does our representation + TaViT generalise?*
(separately from: *Does our auto-segmentation generalise?*)
In production the pipeline generates masks. In validation we use
reference masks to test the representation model cleanly.

**OpenBTAI mask label encoding (verified from full 373-mask audit):**
```
label = lesion_id x 10 + component
  component 1 = Contrast-Enhancing (CE)  -> Cyprus label 3 (ET)
  component 2 = Necrotic core            -> Cyprus label 1 (NCR)
  (Edema NOT annotated in OpenBTAI)
Labels found: 11,12,21,22,...,91,92  (up to 9 lesions per scan)
```

**Multi-lesion handling:**
OpenBTAI has up to 9 simultaneous lesions per scan.
Cyprus-PROTEAS training used 1 dominant lesion.
We select the **largest CE lesion** to match training distribution.

**Steps:**
1. Parse filenames -> openbtai_patient_timelines.csv
2. Map OpenBTAI labels to Cyprus BraTS subregion format {0,1,3}
3. Select dominant (largest CE) lesion per scan
4. Resample to 1mm isotropic
5. Z-score normalise (non-zero brain voxels)
6. Replicate T1c -> 4 channels (BSF expects t1,t1c,t2,flair)
7. Save in Cyprus-compatible structure
8. Compute openbtai_scan_volumes.csv

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import nibabel as nib
import os, json
from pathlib import Path
from datetime import datetime
from scipy.ndimage import zoom
from tqdm import tqdm
print("Imports OK")

Imports OK


In [2]:
# ══════════════════════ CONFIGURATION ════════════════════════════════
SRC_IMAGES = Path('/home/moamed/HDD/validation_data/NIFTIs_Images')
SRC_MASKS  = Path('/home/moamed/HDD/validation_data/NIFTIs_Masks')
OUT_DIR    = Path('/home/moamed/HDD/validation_data/preprocessed_openbtai')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_VOXEL = (1.0, 1.0, 1.0)  # 1mm isotropic — matches Cyprus-PROTEAS
N_CHANNELS   = 4                 # BSF SwinUNETR expects [t1, t1c, t2, flair]

# OpenBTAI label encoding — verified from full data audit:
#   label = lesion_id * 10 + component
#   component 1 = CE  -> Cyprus label 3 (Enhancing Tumor)
#   component 2 = NCR -> Cyprus label 1 (Necrotic Core)
COMP_CE  = 1;  COMP_NCR = 2
CY_NCR   = 1;  CY_ET    = 3

for p, name in [(SRC_IMAGES,'Images'), (SRC_MASKS,'Masks')]:
    n = len(list(p.glob('*.nii')))
    assert p.exists() and n > 0, f"{name} not found: {p}"
    print(f"{name}: {n} files OK")
print(f"Output: {OUT_DIR}")

Images: 373 files OK
Masks: 373 files OK
Output: /home/moamed/HDD/validation_data/preprocessed_openbtai


In [3]:
# ══════════════ PARSE FILENAMES -> TIMELINES ═════════════════════════
# Format: {patient_id}_{date}_{sequence}_img.nii
# Dates are anonymised (shifted to 1900s) — only RELATIVE GAPS are real.

def parse_fn(fname):
    stem  = fname.replace('_img.nii','').replace('_msk.nii','')
    parts = stem.split('_', 2)
    if len(parts) < 2: return None
    return {'patient_id':parts[0], 'date_str':parts[1],
            'sequence':parts[2] if len(parts)>2 else 'unknown'}

def date_ord(d):
    try:
        yr  = int(d[0:4])
        mon = max(1, min(12, int(d[4:6])))
        day = max(1, min(28, int(d[6:8])))
        return datetime(yr, mon, day).toordinal()
    except: return None

records = []
for f in sorted(SRC_IMAGES.glob('*.nii')):
    p = parse_fn(f.name)
    if not p: continue
    p['img_path'] = str(f)
    msk = SRC_MASKS / f.name.replace('_img.nii','_msk.nii')
    p['msk_path'] = str(msk) if msk.exists() else None
    p['has_mask'] = msk.exists()
    records.append(p)

df = pd.DataFrame(records)
df['ordinal'] = df['date_str'].apply(date_ord)
df = df.dropna(subset=['ordinal']).copy()
df['ordinal'] = df['ordinal'].astype(int)
df = df.sort_values(['patient_id','ordinal']).reset_index(drop=True)

base = df.groupby('patient_id')['ordinal'].min().to_dict()
df['days_since_baseline'] = df.apply(lambda r: r['ordinal']-base[r['patient_id']], axis=1)
df['visit_idx']  = df.groupby('patient_id').cumcount()
df['visit_name'] = df['visit_idx'].apply(lambda i: 'baseline' if i==0 else f'fu{i}')

vis = df.groupby('patient_id').size()
print(f"Parsed {len(df)} scans, {df['patient_id'].nunique()} patients")
print(f"Masks: {df['has_mask'].sum()}/{len(df)}")
print(f"Visits/patient: mean={vis.mean():.1f}, min={vis.min()}, max={vis.max()}")
print(df[['patient_id','visit_name','days_since_baseline','sequence']].head(6).to_string(index=False))

Parsed 373 scans, 75 patients
Masks: 373/373
Visits/patient: mean=5.0, min=2, max=10
patient_id visit_name  days_since_baseline        sequence
     10005   baseline                    0    eTRA-3D-T1Gd
     10005        fu1                  115    eTRA-3D-T1Gd
     10005        fu2                  224       TRA-3D-T1
     10005        fu3                  299    eTRA-3D-T1Gd
     10020   baseline                    0    TRA_3D_T1+GD
     10020        fu1                   67 TRA_3D_T1_HR+GD


In [4]:
# ══════════════ MASK + PREPROCESSING FUNCTIONS ═══════════════════════

def map_labels_to_cyprus(mask_data):
    """Map OpenBTAI label encoding to Cyprus BraTS subregion format.
    label = lesion_id * 10 + component
    component 1 = CE  -> Cyprus label 3 (Enhancing Tumor)
    component 2 = NCR -> Cyprus label 1 (Necrotic Core)
    Edema (Cyprus label 2) absent in OpenBTAI — will be 0.
    Returns uint8 mask with values in {0, 1, 3}.
    """
    m   = mask_data.astype(int)
    out = np.zeros_like(mask_data, dtype=np.uint8)
    out[m % 10 == COMP_CE]  = CY_ET   # 3
    out[m % 10 == COMP_NCR] = CY_NCR  # 1
    out[m == 0]              = 0
    return out

def get_dominant_lesion(mask_data):
    """Select the dominant (largest CE) lesion from a multi-lesion OpenBTAI mask.
    Returns: mapped_mask (Cyprus {0,1,3}), lesion_id, n_lesions, ce_voxel_count
    """
    m = mask_data.astype(int)
    lesion_ids = sorted(set(v//10 for v in m.ravel() if v > 0))
    if not lesion_ids:
        return np.zeros_like(mask_data, dtype=np.uint8), 0, 0, 0
    # Pick lesion with most CE voxels
    best_lid, best_ce = -1, -1
    for lid in lesion_ids:
        ce = int(np.sum(m == lid*10 + COMP_CE))
        if ce > best_ce:
            best_ce, best_lid = ce, lid
    dom_raw = np.where(m//10 == best_lid, m, 0).astype(float)
    return map_labels_to_cyprus(dom_raw), best_lid, len(lesion_ids), best_ce

def resample(data, cur_voxel, tgt=(1.0,1.0,1.0), order=1):
    zf = tuple(c/t for c,t in zip(cur_voxel, tgt))
    return zoom(data, zf, order=order, mode='nearest')

def zscore(data):
    """Z-score normalisation within non-zero brain region (BraTS protocol).

    IMPORTANT: We save the original zero mask and restore it after normalisation.
    Without this, background voxels (originally 0) get shifted to negative values
    by the subtraction, making the volume 100% non-zero.
    Cyprus BraTS data is skull-stripped (18% non-zero); restoring zeros ensures
    our normalised output matches that convention.

    Note on skull stripping: OpenBTAI is NOT skull-stripped (65% non-zero raw).
    However, the BSF ROI crop (tumor bbox +16px) removes the skull BEFORE the
    model sees the data, so skull stripping is not required for valid BSF extraction.
    The Z-score correction below is still necessary for correct statistics.
    """
    zero_mask = (data == 0)           # Save background BEFORE normalisation
    nz = data[~zero_mask]
    if len(nz) == 0 or nz.std() == 0: return data.astype(np.float32)
    out = ((data - nz.mean()) / nz.std()).astype(np.float32)
    out[zero_mask] = 0.0              # Restore background zeros - CRITICAL FIX
    return out

def vol_mm3(mask, voxel=(1.0,1.0,1.0)):
    return int(np.count_nonzero(mask)) * float(voxel[0])*float(voxel[1])*float(voxel[2])

print("All preprocessing functions defined.")

All preprocessing functions defined.


In [5]:
# ══════════════ TEST ON ONE SCAN ═════════════════════════════════════
# Patient 10020 has up to 7 simultaneous lesions — good stress test

row = df[df['patient_id']=='10020'].iloc[0]
img_nii = nib.load(row['img_path'])
msk_nii = nib.load(row['msk_path'])
img_raw = img_nii.get_fdata().astype(np.float32)
msk_raw = msk_nii.get_fdata()
voxel   = tuple(float(v) for v in img_nii.header.get_zooms()[:3])

print(f"Test: {row['patient_id']} / {row['visit_name']}")
print(f"Raw labels: {sorted(set(msk_raw.astype(int).ravel())-{0})}")

mapped, lid, n_les, ce_vol = get_dominant_lesion(msk_raw)
print(f"Lesions: {n_les}, selected lesion {lid} (CE voxels: {ce_vol:,})")
print(f"Mapped unique: {np.unique(mapped).tolist()}  ET:{np.sum(mapped==3):,}  NCR:{np.sum(mapped==1):,}")

img_res = resample(img_raw, voxel, TARGET_VOXEL, order=1)
msk_res = resample(mapped.astype(float), voxel, TARGET_VOXEL, order=0)
msk_res = np.round(msk_res).astype(np.uint8)
img_norm = zscore(img_res)
# Single channel image (replicated to 4ch by BSF loader at inference time)
print(f'Normalised image shape: {img_norm.shape}  dtype={img_norm.dtype}')
print(f"After resample: img={img_res.shape}, mask unique={np.unique(msk_res).tolist()}")
print(f"Z-score: mean={img_norm[img_norm!=0].mean():.3f}, std={img_norm[img_norm!=0].std():.3f}")
print(f"Single-channel shape (float32): {img_norm.shape}  ready for BSF SwinUNETR")
vol_b = vol_mm3(mapped>0, voxel)
vol_a = vol_mm3(msk_res>0, TARGET_VOXEL)
print(f"WT volume: {vol_b:.0f} -> {vol_a:.0f} mm3 ({abs(vol_a-vol_b)/max(1,vol_b)*100:.1f}% delta)")
print("Test passed OK")

Test: 10020 / baseline
Raw labels: [np.int64(11), np.int64(12), np.int64(21), np.int64(22), np.int64(31), np.int64(32), np.int64(41), np.int64(42), np.int64(51), np.int64(52), np.int64(61), np.int64(62), np.int64(71), np.int64(72)]
Lesions: 7, selected lesion 6 (CE voxels: 12,761)
Mapped unique: [0, 1, 3]  ET:12,761  NCR:1,271
Normalised image shape: (230, 230, 128)  dtype=float32
After resample: img=(230, 230, 128), mask unique=[0, 1, 3]
Z-score: mean=0.000, std=1.000
Single-channel shape (float32): (230, 230, 128)  ready for BSF SwinUNETR
WT volume: 2265 -> 2254 mm3 (0.5% delta)
Test passed OK


## Main Loop — Process All 373 Scans

Output structure (mirrors Cyprus-PROTEAS BraTS layout for BSF compatibility):
```
preprocessed_openbtai/
  {patient_id}/{visit_name}/
    image_4ch.nii.gz         <- (H, W, D, 4) float32 normalised
    mask_subregions.nii.gz   <- (H, W, D) uint8 {0, 1, 3}
```
The `mask_subregions.nii.gz` is passed to BSF's ROI crop:
`wt_mask = labels_tensor[0,0] > 0` — any non-zero = tumour boundary.

In [6]:
# ══════════════ MAIN LOOP — PROCESS ALL 373 SCANS ════════════════════
volume_records = []
lesion_records = []
failed_scans   = []
processed      = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing"):
    pid, visit = row['patient_id'], row['visit_name']
    scan_dir   = OUT_DIR / pid / visit
    scan_dir.mkdir(parents=True, exist_ok=True)
    out_img = scan_dir / 'image_t1c.nii.gz'
    out_msk = scan_dir / 'mask_subregions.nii.gz'

    if out_img.exists() and out_msk.exists():
        try:
            ex = nib.load(str(out_msk)).get_fdata()
            volume_records.append({'patient_id':pid,'visit_name':visit,
                'vol_ET_mm3':round(vol_mm3(ex==3),1),
                'vol_NCR_mm3':round(vol_mm3(ex==1),1),
                'vol_WT_mm3':round(vol_mm3(ex>0),1)})
        except: pass
        processed += 1; continue

    try:
        img_nii = nib.load(row['img_path'])
        msk_nii = nib.load(row['msk_path'])
        img_raw = img_nii.get_fdata().astype(np.float32)
        msk_raw = msk_nii.get_fdata()
        voxel   = tuple(float(v) for v in img_nii.header.get_zooms()[:3])

        if img_raw.shape != msk_raw.shape:
            raise ValueError(f"Shape mismatch {img_raw.shape} vs {msk_raw.shape}")
        if any(v<=0 or v>15 for v in voxel):
            raise ValueError(f"Bad voxel {voxel}")

        mapped, lid, n_les, ce_vol = get_dominant_lesion(msk_raw)
        lesion_records.append({'patient_id':pid,'visit_name':visit,
            'n_lesions':n_les,'dominant_lesion_id':lid,'dominant_ce_voxels':ce_vol})

        img_res  = resample(img_raw, voxel, TARGET_VOXEL, order=1)
        msk_res  = np.round(resample(mapped.astype(float), voxel, TARGET_VOXEL, order=0)).astype(np.uint8)
        img_norm = zscore(img_res)

        volume_records.append({'patient_id':pid,'visit_name':visit,
            'vol_ET_mm3':round(vol_mm3(msk_res==3),1),
            'vol_NCR_mm3':round(vol_mm3(msk_res==1),1),
            'vol_WT_mm3':round(vol_mm3(msk_res>0),1)})

        affine = np.eye(4)
        # Save single-channel float32 — explicit header ensures float32 on disk (~12MB vs 94MB)
        hdr32 = nib.Nifti1Header(); hdr32.set_data_dtype(np.float32)
        nib.save(nib.Nifti1Image(img_norm.astype(np.float32), affine, hdr32), str(out_img))
        nib.save(nib.Nifti1Image(msk_res, affine), str(out_msk))
        processed += 1

    except Exception as e:
        failed_scans.append((pid, visit, str(e)))

print(f"Processed: {processed}/{len(df)}")
print(f"Failed:    {len(failed_scans)}")
for pid,visit,reason in failed_scans[:5]:
    print(f"  {pid}/{visit}: {reason}")

Preprocessing:   0%|          | 0/373 [00:00<?, ?it/s]

Preprocessing: 100%|██████████| 373/373 [35:45<00:00,  5.75s/it]  

Processed: 373/373
Failed:    0


In [7]:
# ══════════════ SAVE ALL OUTPUTS ═════════════════════════════════════
tcols = ['patient_id','visit_name','visit_idx','days_since_baseline','date_str','sequence','has_mask']
df[tcols].to_csv(OUT_DIR/'openbtai_patient_timelines.csv', index=False)
print(f"Saved: openbtai_patient_timelines.csv ({len(df)} rows)")

vol_df = pd.DataFrame(volume_records)
vol_df.to_csv(OUT_DIR/'openbtai_scan_volumes.csv', index=False)
print(f"Saved: openbtai_scan_volumes.csv ({len(vol_df)} rows)")

les_df = pd.DataFrame(lesion_records)
les_df.to_csv(OUT_DIR/'openbtai_lesion_selection.csv', index=False)
n_multi = int((les_df['n_lesions']>1).sum()) if len(les_df) else 0
print(f"Saved: openbtai_lesion_selection.csv ({len(les_df)} rows, {n_multi} multi-lesion scans)")

meta = {
    'source_dataset':  'OpenBTAI (Ocana-Tienda et al., Scientific Data 2023)',
    'n_patients':      int(df['patient_id'].nunique()),
    'n_scans':         len(df), 'n_processed': processed, 'n_failed': len(failed_scans),
    'target_voxel_mm': list(TARGET_VOXEL), 'n_channels': 1,
    'normalisation':   'z-score within non-zero brain voxels',
    'mask_encoding':   'OpenBTAI label=lesion_id*10+comp mapping: CE->label3, NCR->label1. Edema absent.',
    'lesion_selection':'dominant = lesion with most CE voxels per scan',
    'channel_strategy':'T1c single channel float32 (replicated to 4ch at BSF load time)',
    'mask_role':       'Expert ground-truth masks used directly as BSF input to isolate representation quality',
    'failed_scans':    failed_scans,
}
with open(OUT_DIR/'preprocessing_metadata.json','w') as f: json.dump(meta,f,indent=2)
print("Saved: preprocessing_metadata.json")

Saved: openbtai_patient_timelines.csv (373 rows)
Saved: openbtai_scan_volumes.csv (373 rows)
Saved: openbtai_lesion_selection.csv (143 rows, 52 multi-lesion scans)
Saved: preprocessing_metadata.json


In [8]:
# ══════════════ QUALITY REPORT ═══════════════════════════════════════
print("="*65)
print("  VALIDATION STEP 1 — PREPROCESSING QUALITY REPORT")
print("="*65)

total_imgs = sum(1 for _ in OUT_DIR.rglob('image_t1c.nii.gz'))
total_msks = sum(1 for _ in OUT_DIR.rglob('mask_subregions.nii.gz'))
print(f"\nOutput counts:")
print(f"  image_t1c.nii.gz:       {total_imgs}")
print(f"  mask_subregions.nii.gz: {total_msks}")
print(f"  Paired scans:           {min(total_imgs,total_msks)}")

if len(vol_df) > 0:
    print(f"\nVolume stats (mm3):")
    for col, lab in [('vol_ET_mm3','ET'),('vol_NCR_mm3','NCR'),('vol_WT_mm3','WT')]:
        s = vol_df[col]
        print(f"  {lab}: mean={s.mean():.0f}  median={s.median():.0f}  max={s.max():.0f}  zeros={(s==0).sum()}")

if len(les_df) > 0:
    print(f"\nLesion selection:")
    print(f"  Single-lesion: {(les_df['n_lesions']==1).sum()}")
    print(f"  Multi-lesion:  {(les_df['n_lesions']>1).sum()}")
    print(f"  Max lesions:   {les_df['n_lesions'].max()}")

sample_list = list(OUT_DIR.rglob('image_t1c.nii.gz'))
if sample_list:
    s = sample_list[0]
    si = nib.load(str(s))
    sm = nib.load(str(s.parent/'mask_subregions.nii.gz'))
    d,m = si.get_fdata(), sm.get_fdata()
    print(f"\nSpot-check: {s.relative_to(OUT_DIR)}")
    print(f"  image shape: {si.shape}  (expect H,W,D,4)")
    print(f"  dtype: {d.dtype},  range: [{d.min():.2f},{d.max():.2f}]")
    print(f"  mask unique: {np.unique(m).astype(int).tolist()}  (expect subset of [0,1,3])")

print(f"\n{'='*65}")
status = "ALL SCANS PREPROCESSED SUCCESSFULLY OK" if not failed_scans else f"{len(failed_scans)} SCANS FAILED"
print(f"  {status}")
print("="*65)
print("\nNEXT: Validation_Step2_BSF_Extraction.ipynb")
print("  -> Upload preprocessed_openbtai/ + frozen BSF weights to Kaggle")
print("  -> Run FROZEN BSF v2 extraction (inference, no retraining)")

# ═══════════════════ CHANNEL STRATEGY NOTE ═══════════════════════════
print()
print("=== 4-CHANNEL STRATEGY: REPLICATION (T1c x4) ===")
print("OpenBTAI has T1c only. BSF SwinUNETR expects 4 channels.")
print("We replicate T1c to all 4 slots: [T1c, T1c, T1c, T1c]")
print()
print("Justification:")
print("  Cyprus modality correlations with T1c:")
print("    T1: 0.975, T2: 0.892, FLAIR: 0.938")
print("  The 4 training channels were already highly correlated.")
print("  Replication introduces redundancy but NOT catastrophic domain shift.")
print("  This is standard practice in literature for single-modality adaptation.")
print()
print("Alternative (tested): [0, T1c, 0, 0] — zero-fill missing channels")
print("  Risk: 75% of input dead -> BatchNorm stats mismatch")
print("  Not recommended without fine-tuning.")
print()
print("=> Documented as a validation limitation in the report.")

print()
print("=== SKULL STRIPPING NOTE ===")
print("Cyprus BraTS data: skull-stripped (18% non-zero)")
print("OpenBTAI raw:      NOT skull-stripped (65% non-zero)")
print("Impact on BSF extraction: NONE")
print("  Reason: BSF ROI crop (tumor bbox + 16px padding) runs BEFORE model.")
print("  The skull is fully removed by the crop. BSF never sees the skull.")
print("Impact on Z-score: FIXED by restoring zero mask after normalisation.")
print()
print("Conclusion: Skull stripping is NOT required for valid BSF extraction.")
print("           The Z-score fix (restoring zero mask) is sufficient.")

  VALIDATION STEP 1 — PREPROCESSING QUALITY REPORT

Output counts:
  image_t1c.nii.gz:       373
  mask_subregions.nii.gz: 373
  Paired scans:           373

Volume stats (mm3):
  ET: mean=3565  median=1712  max=24959  zeros=0
  NCR: mean=867  median=104  max=22827  zeros=70
  WT: mean=4432  median=1997  max=29513  zeros=0

Lesion selection:
  Single-lesion: 91
  Multi-lesion:  52
  Max lesions:   4

Spot-check: 20013/baseline/image_t1c.nii.gz
  image shape: (230, 230, 160)  (expect H,W,D,4)
  dtype: float64,  range: [-0.96,9.07]
  mask unique: [0, 1, 3]  (expect subset of [0,1,3])

  ALL SCANS PREPROCESSED SUCCESSFULLY OK

NEXT: Validation_Step2_BSF_Extraction.ipynb
  -> Upload preprocessed_openbtai/ + frozen BSF weights to Kaggle
  -> Run FROZEN BSF v2 extraction (inference, no retraining)

=== 4-CHANNEL STRATEGY: REPLICATION (T1c x4) ===
OpenBTAI has T1c only. BSF SwinUNETR expects 4 channels.
We replicate T1c to all 4 slots: [T1c, T1c, T1c, T1c]

Justification:
  Cyprus modality co